In [0]:
%pip install torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 426.4/426.4 MB 112.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.6/444.6 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.1/221.1 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 80.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.2/60.2 MB 94.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.5/188.5 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 93.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [0]:
dbutils.library.restartPython()

In [0]:
import pandas as pd
import numpy as np

DATASET_PATH = "/Volumes/aml_pipeline/transactions/raw_data/elliptic_bitcoin_dataset"

# -- Load all 3 files ----------------------------------------
print("Loading Elliptic dataset...")

# Features — 166 anonymous features per transaction
features_df = pd.read_csv(
    f"{DATASET_PATH}/elliptic_txs_features.csv",
    header=None
)
# Name the columns properly
feature_cols = ["txId", "time_step"] + [f"feature_{i}" for i in range(1, 166)]
features_df.columns = feature_cols

# Edge list — connections between transactions
edges_df = pd.read_csv(
    f"{DATASET_PATH}/elliptic_txs_edgelist.csv"
)

# Class labels — illicit=1, licit=2, unknown
classes_df = pd.read_csv(
    f"{DATASET_PATH}/elliptic_txs_classes.csv"
)

print("Dataset loaded successfully!")
print(f"\nFeatures shape  : {features_df.shape}")
print(f"Edges shape     : {edges_df.shape}")
print(f"Classes shape   : {classes_df.shape}")

# -- Explore the labels --------------------------------------
print(f"\nLabel distribution:")
print(classes_df["class"].value_counts())

# -- Merge features with labels ------------------------------
data_df = features_df.merge(classes_df, on="txId", how="left")

# Keep only labeled transactions (drop unknowns for training)
labeled_df = data_df[data_df["class"] != "unknown"].copy()
labeled_df["label"] = (labeled_df["class"] == "1").astype(int)

print(f"\nLabeled transactions : {len(labeled_df):,}")
print(f"  Illicit (fraud)    : {(labeled_df['label']==1).sum():,} ({round((labeled_df['label']==1).mean()*100,1)}%)")
print(f"  Licit (clean)      : {(labeled_df['label']==0).sum():,} ({round((labeled_df['label']==0).mean()*100,1)}%)")

print(f"\nSample of labeled data:")
print(labeled_df[["txId", "time_step", "feature_1", "feature_2", "label"]].head())

Loading Elliptic dataset...
Dataset loaded successfully!

Features shape  : (203769, 167)
Edges shape     : (234355, 2)
Classes shape   : (203769, 2)

Label distribution:
class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64

Labeled transactions : 46,564
  Illicit (fraud)    : 4,545 (9.8%)
  Licit (clean)      : 42,019 (90.2%)

Sample of labeled data:
         txId  time_step  feature_1  feature_2  label
3   232438397          1   0.163054   1.963790      0
9   232029206          1  -0.005027   0.578941      0
10  232344069          1  -0.147852  -0.184668      0
11   27553029          1  -0.151357  -0.184668      0
16    3881097          1  -0.172306  -0.184668      0


In [0]:
import numpy as np
import pandas as pd
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_score, recall_score, f1_score
)
from sklearn.model_selection import train_test_split
from mlflow.models.signature import infer_signature

print("All libraries loaded")

# ── STEP 1: Prepare node features ───────────────────────────
print("\nSTEP 1: Preparing features...")

feature_cols = [f"feature_{i}" for i in range(1, 166)]

# Use only labeled transactions
X = labeled_df[feature_cols].values.astype(np.float32)
y = labeled_df["label"].values.astype(np.float32)
node_ids = labeled_df["txId"].values

# Normalize features — GNNs train better on normalized data
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Train/test split — 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")
print(f"Features per node: {X_train.shape[1]}")

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

# ── STEP 2: Define the GNN model ────────────────────────────
# We use a simple but effective 3-layer neural network
# that mimics GraphSAGE's aggregation pattern.
# In a full production system this would use PyTorch Geometric
# but we keep it simple here for Databricks compatibility.

class FraudGNN(nn.Module):
    """
    3-layer fraud detection network.
    Layer 1: Input features -> 128 hidden units
    Layer 2: 128 -> 64 hidden units
    Layer 3: 64 -> 1 output (fraud probability)
    Dropout prevents overfitting on the small fraud class.
    """
    def __init__(self, input_dim, hidden_dim=128, dropout=0.3):
        super(FraudGNN, self).__init__()
        self.layer1   = nn.Linear(input_dim, hidden_dim)
        self.layer2   = nn.Linear(hidden_dim, 64)
        self.layer3   = nn.Linear(64, 1)
        self.dropout  = nn.Dropout(dropout)
        self.bn1      = nn.BatchNorm1d(hidden_dim)
        self.bn2      = nn.BatchNorm1d(64)

    def forward(self, x):
        x = F.relu(self.bn1(self.layer1(x)))
        x = self.dropout(x)
        x = F.relu(self.bn2(self.layer2(x)))
        x = self.dropout(x)
        x = torch.sigmoid(self.layer3(x))
        return x.squeeze()

# ── STEP 3: Train with MLflow tracking ──────────────────────
print("\nSTEP 2: Training GNN model with MLflow tracking...")

# Hyperparameters
EPOCHS      = 50
BATCH_SIZE  = 512
LR          = 0.001
HIDDEN_DIM  = 128
DROPOUT     = 0.3

# Class weights — fraud (1) is rare so we weight it higher
# This prevents the model from just predicting everything as clean
fraud_weight = (y_train == 0).sum() / (y_train == 1).sum()
pos_weight   = torch.tensor([fraud_weight], dtype=torch.float32)

mlflow.set_experiment("/sentinelflow-gnn-fraud-detection")

with mlflow.start_run(run_name="fraud_gnn_v1"):

    # Log hyperparameters
    mlflow.log_params({
        "epochs":      EPOCHS,
        "batch_size":  BATCH_SIZE,
        "learning_rate": LR,
        "hidden_dim":  HIDDEN_DIM,
        "dropout":     DROPOUT,
        "model_type":  "GraphSAGE-style FraudGNN",
        "dataset":     "Elliptic Bitcoin Dataset",
        "train_size":  len(X_train),
        "test_size":   len(X_test),
        "fraud_weight": round(float(fraud_weight), 2),
    })

    # Initialize model
    model    = FraudGNN(input_dim=X_train.shape[1], hidden_dim=HIDDEN_DIM, dropout=DROPOUT)
    optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-4)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    # Training loop
    model.train()
    for epoch in range(EPOCHS):
        # Mini-batch training
        perm        = torch.randperm(len(X_train_t))
        epoch_loss  = 0
        num_batches = 0

        for i in range(0, len(X_train_t), BATCH_SIZE):
            idx        = perm[i:i+BATCH_SIZE]
            batch_X    = X_train_t[idx]
            batch_y    = y_train_t[idx]

            optimizer.zero_grad()
            out  = model(batch_X).squeeze()
            loss = criterion(out, batch_y)
            loss.backward()
            optimizer.step()

            epoch_loss  += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches

        # Log loss every 10 epochs
        if (epoch + 1) % 10 == 0:
            mlflow.log_metric("train_loss", avg_loss, step=epoch+1)
            print(f"  Epoch {epoch+1:>3}/{EPOCHS} | Loss: {avg_loss:.4f}")

    # ── Evaluate on test set ─────────────────────────────────
    print("\nSTEP 3: Evaluating model...")
    model.eval()
    with torch.no_grad():
        test_probs = model(X_test_t).numpy()
        test_preds = (test_probs > 0.5).astype(int)

    auc       = roc_auc_score(y_test, test_probs)
    precision = precision_score(y_test, test_preds, zero_division=0)
    recall    = recall_score(y_test, test_preds, zero_division=0)
    f1        = f1_score(y_test, test_preds, zero_division=0)

    # Log metrics to MLflow
    mlflow.log_metrics({
        "test_auc":       round(auc, 4),
        "test_precision": round(precision, 4),
        "test_recall":    round(recall, 4),
        "test_f1":        round(f1, 4),
    })

    print(f"\n  AUC Score   : {auc:.4f}")
    print(f"  Precision   : {precision:.4f}")
    print(f"  F1 Score    : {f1:.4f}")
    print(f"  Recall      : {recall:.4f}")

    print(f"\n  Classification Report:")
    print(classification_report(y_test, test_preds,
          target_names=["Licit", "Illicit"]))

    # ── Save model to MLflow ─────────────────────────────────
    # Create signature from test data
    sample_input  = pd.DataFrame(X_test[:5], columns=[f"feature_{i}" for i in range(1, 166)])
    sample_output = pd.DataFrame(test_probs[:5], columns=["fraud_risk_score"])
    signature     = infer_signature(sample_input, sample_output)

    mlflow.pytorch.log_model(
        model,
        "fraud_gnn_model",
        registered_model_name="SentinelFlow_FraudGNN",
        signature=signature
    )
    print("Model saved to MLflow Model Registry!")

print("\nGNN training complete!")

All libraries loaded

STEP 1: Preparing features...
Training samples : 37,251
Test samples     : 9,313
Features per node: 165

STEP 2: Training GNN model with MLflow tracking...
  Epoch  10/50 | Loss: 0.9659
  Epoch  20/50 | Loss: 0.9537
  Epoch  30/50 | Loss: 0.9497
  Epoch  40/50 | Loss: 0.9467


2026/05/20 21:50:02 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Epoch  50/50 | Loss: 0.9438

STEP 3: Evaluating model...

  AUC Score   : 0.9792
  Precision   : 0.8304
  F1 Score    : 0.8612
  Recall      : 0.8944

  Classification Report:
              precision    recall  f1-score   support

       Licit       0.99      0.98      0.98      8404
     Illicit       0.83      0.89      0.86       909

    accuracy                           0.97      9313
   macro avg       0.91      0.94      0.92      9313
weighted avg       0.97      0.97      0.97      9313



🔗 View Logged Model at: https://dbc-b283989d-573d.cloud.databricks.com/ml/experiments/2125284915240247/models/m-17f0ae439a004b91b2cf7b99e40232dc?o=7474657231332603
Registered model 'SentinelFlow_FraudGNN' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

🔗 Created version '1' of model 'workspace.default.sentinelflow_fraudgnn': https://dbc-b283989d-573d.cloud.databricks.com/explore/data/models/workspace/default/sentinelflow_fraudgnn/version/1?o=7474657231332603


Model saved to MLflow Model Registry!

GNN training complete!


In [0]:
import mlflow
import mlflow.pytorch
import torch
import numpy as np
import pandas as pd
from pyspark.sql.functions import col, udf, lit, current_timestamp
from pyspark.sql.types import DoubleType
from sklearn.preprocessing import StandardScaler

SILVER_TABLE = "aml_pipeline.transactions.silver_transactions"
GOLD_TABLE   = "aml_pipeline.transactions.gold_sar_reports"

# -- Step 1: Load trained model from MLflow ------------------
print("Loading trained GNN model from MLflow...")

model_uri = "models:/workspace.default.sentinelflow_fraudgnn/1"
loaded_model = mlflow.pytorch.load_model(model_uri)
loaded_model.eval()
print("Model loaded successfully!")

# -- Step 2: Load Silver table and prepare features ----------
print("\nPreparing features from Silver table...")

# We use the same 166 features from Elliptic
# mapped to our transaction features
# Since our transactions don't have exact Elliptic features,
# we create proxy features from available fields
silver_df = spark.table(SILVER_TABLE)
silver_pd  = silver_df.select(
    "transaction_id",
    "amount_usd",
    "batch_number",
    "high_risk_country",
    "large_transaction",
    "is_flagged"
).toPandas()

print(f"Loaded {len(silver_pd):,} transactions")

# -- Step 3: Build proxy feature matrix ----------------------
# We create 165 features from our transaction data
# to match the model's expected input dimension
np.random.seed(42)
n = len(silver_pd)

# Base features from real transaction data
f1  = (silver_pd["amount_usd"] / silver_pd["amount_usd"].max()).fillna(0).values
f2  = silver_pd["high_risk_country"].astype(float).fillna(0).values
f3  = silver_pd["large_transaction"].astype(float).fillna(0).values
f4  = silver_pd["is_flagged"].astype(float).fillna(0).values
f5  = (silver_pd["batch_number"] / 11.0).fillna(0).values

# Fill remaining 160 features with structured noise
# In production these would be real graph features
np.random.seed(42)
noise = np.random.randn(n, 160) * 0.1

# Combine into full feature matrix
X_score = np.column_stack([f1, f2, f3, f4, f5, noise]).astype(np.float32)

# Normalize
scaler  = StandardScaler()
X_score = scaler.fit_transform(X_score)

print(f"Feature matrix shape: {X_score.shape}")

# -- Step 4: Score in batches --------------------------------
print("\nScoring 100,000 transactions...")

BATCH_SIZE = 10_000
all_scores = []

for i in range(0, len(X_score), BATCH_SIZE):
    batch     = torch.tensor(X_score[i:i+BATCH_SIZE], dtype=torch.float32)
    with torch.no_grad():
        scores = loaded_model(batch).numpy()
    all_scores.extend(scores.tolist())
    print(f"  Scored {min(i+BATCH_SIZE, len(X_score)):,} / {len(X_score):,}")

silver_pd["fraud_risk_score"] = [round(float(s), 4) for s in all_scores]

# -- Step 5: Add scores back to Silver table -----------------
print("\nWriting fraud scores to Silver table...")

scores_spark = spark.createDataFrame(
    silver_pd[["transaction_id", "fraud_risk_score"]]
)

# Join scores back to Silver table
silver_with_scores = (
    spark.table(SILVER_TABLE)
    .join(scores_spark, on="transaction_id", how="left")
)

silver_with_scores.write.format("delta").mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(SILVER_TABLE)

# -- Step 6: Update Gold table with fraud scores -------------
print("Updating Gold SAR reports with fraud scores...")

gold_with_scores = (
    spark.table(GOLD_TABLE)
    .join(scores_spark, on="transaction_id", how="left")
)

gold_with_scores.write.format("delta").mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(GOLD_TABLE)

# -- Step 7: Show results ------------------------------------
print("\nFraud score distribution:")
spark.table(SILVER_TABLE).selectExpr(
    "round(avg(fraud_risk_score), 4) as avg_score",
    "round(min(fraud_risk_score), 4) as min_score",
    "round(max(fraud_risk_score), 4) as max_score",
    "count(case when fraud_risk_score > 0.7 then 1 end) as high_risk_count",
    "count(case when fraud_risk_score > 0.5 then 1 end) as medium_risk_count"
).show()

print("Top 5 highest fraud risk transactions:")
spark.table(GOLD_TABLE) \
    .orderBy("fraud_risk_score", ascending=False) \
    .select("sar_reference", "sender_name", "receiver_name",
            "amount_usd", "flag_reason", "fraud_risk_score") \
    .show(5, truncate=False)

print("GNN scoring complete!")

/local_disk0/.ephemeral_nfs/envs/pythonEnv-8c453614-dcf1-4088-8fdd-4fd132af6bd3/lib/python3.12/site-packages/torch/_vmap_internals.py:9: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  from torch.utils._pytree import _broadcast_to_and_flatten, tree_flatten, tree_unflatten


Loading trained GNN model from MLflow...


Model loaded successfully!

Preparing features from Silver table...
Loaded 100,000 transactions
Feature matrix shape: (100000, 165)

Scoring 100,000 transactions...
  Scored 10,000 / 100,000
  Scored 20,000 / 100,000
  Scored 30,000 / 100,000
  Scored 40,000 / 100,000
  Scored 50,000 / 100,000
  Scored 60,000 / 100,000
  Scored 70,000 / 100,000
  Scored 80,000 / 100,000
  Scored 90,000 / 100,000
  Scored 100,000 / 100,000

Writing fraud scores to Silver table...
Updating Gold SAR reports with fraud scores...

Fraud score distribution:
+---------+---------+---------+---------------+-----------------+
|avg_score|min_score|max_score|high_risk_count|medium_risk_count|
+---------+---------+---------+---------------+-----------------+
|   0.1855|      0.0|      1.0|          16611|            18272|
+---------+---------+---------+---------------+-----------------+

Top 5 highest fraud risk transactions:
+---------------------+----------------+----------------+----------+---------------------

In [0]:
# Verify fraud scores are in both tables
print("Silver table check:")
spark.table("aml_pipeline.transactions.silver_transactions") \
    .selectExpr(
        "count(*) as total_records",
        "count(fraud_risk_score) as scored_records",
        "round(avg(fraud_risk_score), 4) as avg_score",
        "round(max(fraud_risk_score), 4) as max_score",
        "count(case when fraud_risk_score > 0.7 then 1 end) as high_risk"
    ).show()

print("Gold table check:")
spark.table("aml_pipeline.transactions.gold_sar_reports") \
    .selectExpr(
        "count(*) as total_sars",
        "count(fraud_risk_score) as scored_sars",
        "round(avg(fraud_risk_score), 4) as avg_score",
        "round(max(fraud_risk_score), 4) as max_score"
    ).show()

print("Top 5 highest fraud risk:")
spark.table("aml_pipeline.transactions.gold_sar_reports") \
    .orderBy("fraud_risk_score", ascending=False) \
    .select("sar_reference", "sender_name", "amount_usd",
            "flag_reason", "fraud_risk_score") \
    .show(5, truncate=False)

Silver table check:
+-------------+--------------+---------+---------+---------+
|total_records|scored_records|avg_score|max_score|high_risk|
+-------------+--------------+---------+---------+---------+
|       100000|        100000|   0.1855|      1.0|    16611|
+-------------+--------------+---------+---------+---------+

Gold table check:
+----------+-----------+---------+---------+
|total_sars|scored_sars|avg_score|max_score|
+----------+-----------+---------+---------+
|      3356|       3356|   0.1346|      1.0|
+----------+-----------+---------+---------+

Top 5 highest fraud risk:
+---------------------+----------------+----------+---------------------------------------------+----------------+
|sar_reference        |sender_name     |amount_usd|flag_reason                                  |fraud_risk_score|
+---------------------+----------------+----------+---------------------------------------------+----------------+
|SAR-20260520-fc32143a|Raj Sharma      |10149.6   |TRAVEL_R